# Synthetic E-Commerce Customer Behavior Dataset — Complete EDA
> **10K customers, 120K transactions, 25K reviews for ML projects** | [Dataset](https://www.kaggle.com/datasets/lorenzoscaturchio/ecommerce-behavior)

**5,000 rows · 10 columns · `customers.csv`**

## Table of Contents
1. [Setup & Data Loading](#setup)
2. [Data Overview](#overview)
3. [Missing Data Analysis](#missing)
4. [Numeric Distributions](#distributions)
5. [Categorical Analysis](#categorical)
6. [Correlation Analysis](#correlations)
7. [Target Analysis](#target)
8. [Key Findings](#findings)

## 1. Setup & Data Loading <a id='setup'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

import os
DATA_DIR = '/kaggle/input/ecommerce-behavior'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'

df = pd.read_csv(f'{DATA_DIR}/customers.csv')
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

## 2. Data Overview <a id='overview'></a>

In [ ]:
print("Column Types:")
print(df.dtypes.value_counts().to_string())
print(f"\nDuplicate rows: {df.duplicated().sum():,}")
print(f"Total missing values: {df.isnull().sum().sum():,}")
print()
df.describe().round(2)

## 3. Missing Data Analysis <a id='missing'></a>

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('percent', ascending=False)

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.4)))
    colors = ['#e74c3c' if p > 20 else '#f39c12' if p > 5 else '#2ecc71'
              for p in missing_df['percent']]
    bars = ax.barh(missing_df.index, missing_df['percent'], color=colors, alpha=0.85)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Data by Column', fontweight='bold')
    for bar, pct in zip(bars, missing_df['percent']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found — dataset is complete!")

## 4. Numeric Distributions <a id='distributions'></a>

In [ ]:
NUMERIC_COLS = ['age', 'is_churned', 'lifetime_value', 'email_opt_in', 'has_app']

n = len(NUMERIC_COLS)
ncols = min(3, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    ax.hist(df[col].dropna(), bins=40, color='#3498db', alpha=0.8, edgecolor='white', linewidth=0.3)
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_val:.2f}')
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for outlier detection
fig, axes = plt.subplots(1, min(len(NUMERIC_COLS), 6), figsize=(min(len(NUMERIC_COLS), 6) * 3, 5))
if min(len(NUMERIC_COLS), 6) == 1:
    axes = [axes]

for ax, col in zip(axes if hasattr(axes, '__iter__') else [axes], NUMERIC_COLS[:6]):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.6))
    ax.set_title(col, fontweight='bold', fontsize=10)

plt.suptitle('Box Plots — Outlier Detection', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Categorical Analysis <a id='categorical'></a>

In [ ]:
CAT_COLS = ['gender', 'country', 'segment']

n = len(CAT_COLS)
ncols = min(2, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

palette = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6',
           '#1abc9c', '#e67e22', '#34495e', '#16a085', '#c0392b']

for i, col in enumerate(CAT_COLS):
    ax = axes[i]
    counts = df[col].value_counts().head(10)
    bars = ax.barh(counts.index.astype(str), counts.values,
                   color=palette[:len(counts)], alpha=0.85)
    ax.set_title(f'{col} (top {min(10, len(counts))})', fontweight='bold')
    ax.set_xlabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + max(counts) * 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis <a id='correlations'></a>

In [ ]:
CORR_COLS = ['age', 'is_churned', 'lifetime_value', 'email_opt_in', 'has_app']

corr = df[CORR_COLS].corr()

fig, ax = plt.subplots(figsize=(min(12, len(CORR_COLS) + 2), min(10, len(CORR_COLS) + 1)))
mask = np.triu(np.ones_like(corr), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, mask=mask,
            annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Top absolute correlations (excluding self-correlations)
corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        corr_pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("Top 10 Feature Correlations:")
print(f"{'Feature A':<25} {'Feature B':<25} {'Correlation':>12}")
print("-" * 65)
for a, b, r in corr_pairs[:10]:
    arrow = '+' if r > 0 else '-'
    print(f"{a:<25} {b:<25} {arrow}{abs(r):>11.3f}")

## 7. Target Analysis <a id='target'></a>

Detected target column: **`is_churned`**

In [ ]:
target_col = 'is_churned'

print(f"Target: {target_col}")
print(f"Unique values: {df[target_col].nunique()}")
print(f"Null: {df[target_col].isnull().sum()}")
print()

if df[target_col].nunique() <= 20:
    print("Value counts:")
    print(df[target_col].value_counts().to_string())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    counts = df[target_col].value_counts()
    ax1.bar(counts.index.astype(str), counts.values, color='#e74c3c', alpha=0.85)
    ax1.set_title(f'{target_col} Distribution', fontweight='bold')
    ax1.set_xlabel(target_col)
    ax1.set_ylabel('Count')
    for x, y in zip(range(len(counts)), counts.values):
        ax1.text(x, y + max(counts) * 0.01, f'{y:,}', ha='center', fontsize=9)

    if len(counts) == 2:
        ax2.pie(counts.values, labels=counts.index.astype(str), autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'], startangle=90)
        ax2.set_title('Class Balance', fontweight='bold')
    else:
        pcts = (counts / counts.sum() * 100).round(1)
        ax2.barh(pcts.index.astype(str), pcts.values, color='#3498db', alpha=0.85)
        ax2.set_xlabel('Percentage (%)')
        ax2.set_title('Class Proportions', fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print(df[target_col].describe())
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df[target_col].dropna(), bins=50, color='#e74c3c', alpha=0.8, edgecolor='white')
    ax.axvline(df[target_col].median(), color='navy', linestyle='--',
               label=f"Median: {df[target_col].median():.2f}")
    ax.set_title(f'{target_col} Distribution', fontweight='bold')
    ax.set_xlabel(target_col)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. Key Findings <a id='findings'></a>

In [ ]:
# Data quality summary
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"  Rows:            {len(df):,}")
print(f"  Columns:         {df.shape[1]}")
print(f"  Duplicates:      {df.duplicated().sum():,}")
print(f"  Total missing:   {df.isnull().sum().sum():,}")
print(f"  Complete rows:   {df.dropna().shape[0]:,} ({df.dropna().shape[0]/len(df)*100:.1f}%)")
print()

numeric_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns
print(f"  Numeric columns:     {len(numeric_cols)}")
print(f"  Categorical columns: {len(cat_cols)}")
print()

if len(numeric_cols) > 0:
    print("Potential outliers (values > 3 std from mean):")
    for col in numeric_cols[:10]:
        z = (df[col] - df[col].mean()) / df[col].std()
        outliers = (z.abs() > 3).sum()
        if outliers > 0:
            print(f"  {col}: {outliers:,} rows ({outliers/len(df)*100:.1f}%)")

### Next Steps
- Feature engineering: create interaction terms from correlated features
- Handle missing values: imputation strategy depends on missingness pattern (MCAR/MAR/MNAR)
- Scale numeric features before modeling (StandardScaler or RobustScaler for outliers)
- Encode categoricals: one-hot for low cardinality, target encoding for high cardinality
- Try baseline models: LogisticRegression/Ridge for linear, XGBoost/LightGBM for tree-based